# 18.1 REST and JSON — Talking to an API

**Prerequisites:** 11.5 HTTP - From Sockets to requests, 8.3 JSON, 2.5 Dictionaries  
**Target:** Python 3.12+ (notes flag 3.13/3.14 differences)

### What you'll learn
- What **REST** actually means in practice: resources, verbs, representations
- `GET`, `POST`, `PUT`, `PATCH`, `DELETE` — and 🔴 which are **safe** and **idempotent**
- The status-code taxonomy that decides what your code should *do*
- 🔴 `raise_for_status()` vs `.ok` vs checking by hand
- Sending and receiving JSON — `json=` vs `data=`, and when `.json()` explodes
- 🔴 The response is **not always JSON**: `204`, and HTML error pages
- Query parameters, headers and content negotiation
- `Session`, and the exception taxonomy `requests` raises

---

## Where this starts

**11.5** built HTTP from the socket up: the request line, headers, `Content-Length`, then
`urllib.request`, then `requests` with timeouts and a `Session`. That was the **transport**.

This folder is about the **conversation** — the conventions almost every web API follows, and
the failures you must handle because they are someone else's server.

### 🔴 Everything here runs offline

A tutorial that calls a real API breaks when that service changes, rate-limits you, or is down
— and it needs credentials nobody should put in a notebook. So this folder runs a **real HTTP
server on localhost**, exactly as **11.5** did. Real sockets, real status codes, real headers,
no network and no secrets.

The server below is the one used throughout this notebook. Read it once; it is a small but
honest imitation of a job-queue API.

In [ ]:
# ---- A fake API, running locally in this notebook ----
# 🔴 Every notebook in this folder runs OFFLINE. Hitting a real API in a
# tutorial makes it break when the service changes, rate-limits you, or is
# simply down - and it would need credentials. A local http.server gives
# real HTTP over a real socket (11.5) with none of that.
import http.server
import json
import socket
import threading
import time
import urllib.parse

JOBS = [
    {"id": f"build-{i:03d}",
     "state": ["queued", "running", "done"][i % 3],
     "attempts": i % 4,
     "region": ["eu", "us", "ap"][i % 3]}
    for i in range(47)
]

REQUEST_LOG = []


class JobsAPI(http.server.BaseHTTPRequestHandler):
    protocol_version = "HTTP/1.1"

    def log_message(self, *args):
        """Silence the default stderr logging."""

    def _send(self, status, payload=None, headers=None):
        extra = headers or {}
        if payload is None:                      # 🔴 204 carries NO body
            self.send_response(status)
            for key, value in extra.items():
                self.send_header(key, value)
            self.send_header("Content-Length", "0")
            self.end_headers()
            return
        body = json.dumps(payload).encode("utf-8")
        self.send_response(status)
        self.send_header("Content-Type", "application/json")
        self.send_header("Content-Length", str(len(body)))
        for key, value in extra.items():
            self.send_header(key, value)
        self.end_headers()
        self.wfile.write(body)

    def _read_json(self):
        length = int(self.headers.get("Content-Length", 0))
        raw = self.rfile.read(length) if length else b""
        return json.loads(raw) if raw else {}

    def do_GET(self):
        parsed = urllib.parse.urlparse(self.path)
        query = urllib.parse.parse_qs(parsed.query)
        REQUEST_LOG.append(("GET", self.path, dict(self.headers)))

        if parsed.path == "/health":
            return self._send(200, {"status": "ok", "version": "1.4.0"})

        if parsed.path == "/echo":
            return self._send(200, {"you_sent": {
                "path": self.path,
                "query": {k: v[0] for k, v in query.items()},
                "accept": self.headers.get("Accept"),
                "user_agent": self.headers.get("User-Agent"),
            }})

        if parsed.path == "/jobs":
            region = query.get("region", [None])[0]
            items = [j for j in JOBS if region is None or j["region"] == region]
            return self._send(200, {"total": len(items), "items": items[:5]})

        if parsed.path.startswith("/jobs/"):
            job_id = parsed.path.rsplit("/", 1)[-1]
            for job in JOBS:
                if job["id"] == job_id:
                    return self._send(200, job)
            return self._send(404, {"error": "not found", "id": job_id})

        if parsed.path == "/html-error":
            body = b"<html><body><h1>502 Bad Gateway</h1></body></html>"
            self.send_response(502)
            self.send_header("Content-Type", "text/html")
            self.send_header("Content-Length", str(len(body)))
            self.end_headers()
            self.wfile.write(body)
            return

        if parsed.path == "/slow":
            time.sleep(2.0)
            return self._send(200, {"eventually": True})

        return self._send(404, {"error": "not found", "path": parsed.path})

    def do_POST(self):
        REQUEST_LOG.append(("POST", self.path, dict(self.headers)))
        try:
            payload = self._read_json()
        except json.JSONDecodeError:
            return self._send(400, {"error": "invalid JSON"})
        if "region" not in payload:
            return self._send(422, {"error": "validation failed",
                                    "detail": [{"field": "region",
                                                "message": "field required"}]})
        return self._send(201, {"id": "build-999", "state": "queued", **payload},
                          {"Location": "/jobs/build-999"})

    def do_PUT(self):
        REQUEST_LOG.append(("PUT", self.path, dict(self.headers)))
        payload = self._read_json()
        return self._send(200, {"id": self.path.rsplit("/", 1)[-1],
                                "replaced": True, **payload})

    def do_PATCH(self):
        REQUEST_LOG.append(("PATCH", self.path, dict(self.headers)))
        payload = self._read_json()
        return self._send(200, {"id": self.path.rsplit("/", 1)[-1],
                                "patched_fields": sorted(payload)})

    def do_DELETE(self):
        REQUEST_LOG.append(("DELETE", self.path, dict(self.headers)))
        return self._send(204)                   # no body, by design


class QuietServer(http.server.ThreadingHTTPServer):
    daemon_threads = True

    def handle_error(self, *args):
        """A client hanging up is normal here, not a crash."""


def start_api():
    probe = socket.socket()
    probe.bind(("127.0.0.1", 0))
    port = probe.getsockname()[1]
    probe.close()
    server = QuietServer(("127.0.0.1", port), JobsAPI)
    threading.Thread(target=server.serve_forever, daemon=True).start()
    return server, f"http://127.0.0.1:{port}"


SERVER, BASE = start_api()
print("fake API listening on", BASE)

## REST, minus the mysticism

"REST" in everyday use means four things:

1. **Resources have URLs.** `/jobs` is the collection, `/jobs/build-007` is one member.
2. **The verb says what you are doing.** Not the URL — `/deleteJob?id=7` is the anti-pattern.
3. **Representations are JSON** (usually), in and out.
4. **The status code carries the outcome**, not just a `{"success": false}` in the body.

```
   GET    /jobs             list the collection
   GET    /jobs/build-007   fetch one member
   POST   /jobs             create a new member       -> 201 + Location
   PUT    /jobs/build-007   replace it entirely
   PATCH  /jobs/build-007   change some fields
   DELETE /jobs/build-007   remove it                 -> 204, no body
```

🔴 **Two properties decide how your client may behave:**

| | **Safe** (no side effects) | **Idempotent** (repeating is harmless) |
|---|---|---|
| `GET` | ✅ | ✅ |
| `HEAD`, `OPTIONS` | ✅ | ✅ |
| `PUT` | ❌ | ✅ — same result whether sent once or five times |
| `DELETE` | ❌ | ✅ — already gone is still gone |
| `POST` | ❌ | 🔴 **❌ — retrying may create a second thing** |
| `PATCH` | ❌ | 🔴 usually **not** (`{"attempts": "+1"}` is not idempotent) |

That table is the reason **18.3** can retry a `GET` freely and must be careful with a `POST`.

In [ ]:
import requests

session = requests.Session()

print("--- the five verbs against the same resource ---")
for label, call in [
    ("GET    /jobs/build-007",
     lambda: session.get(f"{BASE}/jobs/build-007", timeout=5)),
    ("POST   /jobs",
     lambda: session.post(f"{BASE}/jobs", json={"region": "eu"}, timeout=5)),
    ("PUT    /jobs/build-007",
     lambda: session.put(f"{BASE}/jobs/build-007",
                         json={"state": "done", "attempts": 3, "region": "eu"}, timeout=5)),
    ("PATCH  /jobs/build-007",
     lambda: session.patch(f"{BASE}/jobs/build-007", json={"state": "failed"}, timeout=5)),
    ("DELETE /jobs/build-007",
     lambda: session.delete(f"{BASE}/jobs/build-007", timeout=5)),
]:
    response = call()
    location = response.headers.get("Location", "")
    body = response.text[:66] if response.text else "(empty body)"
    print(f"  {label:24} -> {response.status_code} {response.reason:<22} {body}")
    if location:
        print(f"  {'':24}    Location: {location}")

Read the last two lines carefully.

**`POST` returned `201 Created` with a `Location` header** — that is the convention: the server
tells you where the new resource lives, rather than making you guess.

**`DELETE` returned `204 No Content` with an empty body.** Calling `.json()` on that response
raises, which is the next section.

## Status codes: the taxonomy that matters

Memorising all sixty is pointless. What matters is **what your code should do**:

| Range | Meaning | Your move |
|---|---|---|
| **2xx** | it worked | carry on |
| **3xx** | redirect | `requests` follows these by default |
| **4xx** | 🔴 **you** got it wrong | **do not retry** — fix the request |
| **5xx** | 🔴 **they** got it wrong | **retry, with backoff** (**18.3**) |

The ones worth knowing by number:

| Code | Means | Note |
|---|---|---|
| `200` / `201` / `204` | OK / Created / No Content | `201` should carry `Location` |
| `301` / `302` / `307` | moved | 🔴 `307`/`308` preserve the method; `301`/`302` may not |
| `400` | malformed request | your JSON or params are wrong |
| `401` | not authenticated | **18.2** — no credentials, or bad ones |
| `403` | authenticated, not allowed | 🔴 a *different* problem from `401` |
| `404` | no such resource | |
| `409` | conflict | duplicate, or a concurrent edit |
| `422` | well-formed but semantically invalid | the validation failure — **18.4** |
| `429` | rate limited | 🔴 back off — **18.3** |
| `500` / `502` / `503` / `504` | server error / bad gateway / unavailable / timeout | retry |

🔴 **`401` vs `403` is the distinction people get wrong.** `401` means *"I do not know who you
are"* — retrying with a refreshed token may help. `403` means *"I know exactly who you are and
the answer is no"* — retrying is pointless.

## 🔴 Three ways to check a response, two of them wrong

In [ ]:
def check_with_ok(response):
    if response.ok:                     # 🔴 True for ANY 2xx or 3xx
        return "treated as success"
    return f"treated as failure ({response.status_code})"


def check_by_hand(response):
    if response.status_code == 200:     # 🔴 rejects 201 and 204
        return "treated as success"
    return f"treated as failure ({response.status_code})"


def check_with_raise(response):
    try:
        response.raise_for_status()     # raises on 4xx and 5xx only
        return "treated as success"
    except requests.HTTPError as exc:
        return f"raised: {exc.response.status_code}"


created = session.post(f"{BASE}/jobs", json={"region": "eu"}, timeout=5)
deleted = session.delete(f"{BASE}/jobs/build-001", timeout=5)
missing = session.get(f"{BASE}/nope", timeout=5)

print(f"{'response':22}{'.ok':<26}{'== 200':<26}{'raise_for_status'}")
print("-" * 96)
for label, response in (("201 Created", created),
                        ("204 No Content", deleted),
                        ("404 Not Found", missing)):
    print(f"{label:22}{check_with_ok(response):<26}"
          f"{check_by_hand(response):<26}{check_with_raise(response)}")

Read the middle column: **`== 200` rejects a perfectly successful `201` and
`204`.** That is the bug you write when you have only ever seen `GET` succeed.

🔴 **`raise_for_status()` is the right default.** It turns a failure into an exception you
cannot accidentally ignore — the same argument **15.7** made about `print` versus raising.

> **`.ok` is a trap in the other direction**: it is `True` for **3xx** as well, so a redirect
> you did not expect reads as success.

The pattern to internalise:

```python
response = session.get(url, timeout=10)
response.raise_for_status()          # 4xx/5xx become exceptions
data = response.json()
```

## Sending JSON: `json=` vs `data=`

These two look interchangeable and are not.

In [ ]:
import json as jsonlib

payload = {"region": "eu", "state": "queued"}

with_json = session.post(f"{BASE}/jobs", json=payload, timeout=5)
with_data = session.post(f"{BASE}/jobs", data=jsonlib.dumps(payload), timeout=5)
with_form = session.post(f"{BASE}/jobs", data=payload, timeout=5)

print("json=payload      ->", with_json.status_code,
      "| Content-Type sent:", REQUEST_LOG[-3][2].get("Content-Type"))
print("data=json.dumps() ->", with_data.status_code,
      "| Content-Type sent:", REQUEST_LOG[-2][2].get("Content-Type"))
print("data=payload      ->", with_form.status_code,
      "| Content-Type sent:", REQUEST_LOG[-1][2].get("Content-Type"))
print()
print("🔴 `json=` serialises AND sets Content-Type: application/json.")
print("   `data=` with a string sends the same bytes but NO Content-Type -")
print("   this server parses it anyway; many real APIs answer 415 instead.")
print("   `data=` with a dict sends a FORM encoding - a different thing")
print("   entirely, and it failed with 400 because the body is not JSON.")

| You write | Body sent | `Content-Type` | Result here |
|---|---|---|---|
| `json=payload` | JSON | 🔴 `application/json`, set for you | `201` |
| `data=json.dumps(payload)` | the same JSON | **none** — the server must guess | `201`, by luck |
| `data=payload` (a dict) | `region=eu&state=queued` | `application/x-www-form-urlencoded` | 🔴 **`400`** |

The third row failed because a form encoding **is not JSON** — the server tried to parse
`region=eu&state=queued` as JSON and gave up. The second row only worked because this server
does not check `Content-Type`; a stricter API answers `415 Unsupported Media Type`.

**Use `json=`.** The other two exist for form posts and for APIs with unusual requirements.

## 🔴 The response is not always JSON

`.json()` raises when the body is not JSON — and there are two very common cases where it is
not, both of which will happen to you in production.

In [ ]:
def read_body(label, response):
    print(f"  {label}")
    print(f"     status  : {response.status_code}")
    print(f"     type    : {response.headers.get('Content-Type', '(none)')}")
    print(f"     length  : {len(response.content)} bytes")
    try:
        print(f"     .json() : {response.json()}")
    except requests.exceptions.JSONDecodeError as exc:
        print(f"     .json() : 🔴 JSONDecodeError - {exc}")


print("Two responses that are NOT JSON:")
print()
read_body("DELETE -> 204 No Content (nothing to parse)",
          session.delete(f"{BASE}/jobs/build-002", timeout=5))
print()
read_body("A gateway error page (HTML, from a proxy in front of the API)",
          session.get(f"{BASE}/html-error", timeout=5))

print()
print("🔴 The HTML one is the nastier case: a load balancer or proxy answered,")
print("   not the API, so you get HTML with a 5xx and no JSON error object.")
print("   Check the status FIRST, and guard .json() - never assume a shape.")

**The `204` case** is easy to forget: a successful `DELETE` has nothing to
parse. **The HTML case** is the one that bites in production — a proxy, load balancer or CDN
answered instead of the API, so your careful error-object parsing gets a page of HTML.

The defensive shape:

```python
response = session.get(url, timeout=10)
response.raise_for_status()
if not response.content:                       # 204 and friends
    return None
if "application/json" not in response.headers.get("Content-Type", ""):
    raise ValueError(f"expected JSON, got {response.headers.get('Content-Type')}")
return response.json()
```

## Query parameters, and 🔴 why you must not build URLs by hand

In [ ]:
# The right way: requests encodes for you.
proper = session.get(f"{BASE}/echo",
                     params={"region": "eu", "q": "build & deploy", "page": 2},
                     timeout=5)
print("params= :", proper.json()["you_sent"]["query"])
print("  URL   :", proper.url)

print()
# 🔴 The wrong way: string concatenation.
broken = session.get(f"{BASE}/echo?region=eu&q=build & deploy&page=2", timeout=5)
print("hand-built:", broken.json()["you_sent"]["query"])
print("  URL   :", broken.url)
print()
print("🔴 The `&` inside the value was read as a SEPARATOR. The query silently")
print("   became three parameters instead of two, and one of them is nonsense.")
print("   With a value containing `=`, `#` or `+`, it gets worse.")

The space became `%20` and the `&` became `%26` when `requests` did the
encoding — and the hand-built URL silently produced a different query.

🔴 **Always pass `params=`.** The same applies to path segments containing user data: use
`urllib.parse.quote()` rather than an f-string.

## Headers and content negotiation

| Header | Why it matters |
|---|---|
| `Accept` | what representations you can handle |
| `Content-Type` | what you are sending (`json=` sets it) |
| `User-Agent` | 🔴 identify yourself — some APIs reject or throttle unknown clients |
| `Authorization` | credentials — **18.2** |
| `If-None-Match` / `ETag` | conditional requests: `304 Not Modified` saves bandwidth |

Set the ones that apply to every request **once, on the `Session`**.

In [ ]:
client = requests.Session()
client.headers.update({
    "Accept": "application/json",
    "User-Agent": "jobkit-client/1.0 (+https://example.com/jobkit)",
})

seen = client.get(f"{BASE}/echo", timeout=5).json()["you_sent"]
print("the server saw:")
print("   Accept      :", seen["accept"])
print("   User-Agent  :", seen["user_agent"])

print()
print("Session headers apply to every request through this client:")
for name, value in client.headers.items():
    print(f"   {name:16} {value}")

> **`Session` also reuses the TCP connection**, which **11.5** measured. On an API you call
> repeatedly that is the single cheapest performance win — and **18.5** revisits it alongside
> concurrency.

## The exception taxonomy

`requests` raises a small family, all inheriting from `RequestException`. Catching the right
one is what lets **18.3** decide whether to retry.

In [ ]:
from requests import exceptions as rex

print("requests exception hierarchy (the ones you handle):")
for exc in (rex.HTTPError, rex.ConnectionError, rex.Timeout,
            rex.ConnectTimeout, rex.ReadTimeout, rex.TooManyRedirects,
            rex.JSONDecodeError):
    parents = " <- ".join(base.__name__ for base in exc.__mro__[1:4])
    print(f"   {exc.__name__:20} {parents}")

print()
print("--- seen in practice ---")

# 1. A connection that cannot be made at all.
try:
    requests.get("http://127.0.0.1:1/health", timeout=2)
except rex.ConnectionError as exc:
    print(f"   ConnectionError : {type(exc).__name__} (nothing listening)")

# 2. A server that answers too slowly.
try:
    session.get(f"{BASE}/slow", timeout=0.3)
except rex.Timeout as exc:
    print(f"   Timeout         : {type(exc).__name__} (read timed out)")

# 3. A response that arrived, but says no.
try:
    session.get(f"{BASE}/nope", timeout=5).raise_for_status()
except rex.HTTPError as exc:
    print(f"   HTTPError       : {exc.response.status_code} - the server ANSWERED")

print()
print("🔴 The distinction that matters for 18.3:")
print("   ConnectionError / Timeout -> the request may not have been processed")
print("   HTTPError 5xx             -> it was processed and failed")
print("   HTTPError 4xx             -> do not retry; fix the request")

In [ ]:
# ---- tidy up ----
SERVER.shutdown()
print("fake API stopped")
print("requests it handled:", len(REQUEST_LOG))
print("nothing was written to disk, and no network was used")

---

## Common Mistakes & Pitfalls

1. 🔴 **Checking `status_code == 200`.** It rejects `201`, `202` and `204`, which are successes. Use `raise_for_status()`.
2. **Using `.ok`.** It is `True` for 3xx as well, so an unexpected redirect reads as success.
3. 🔴 **Calling `.json()` without checking the status or the content type.** A `204` has no body and a proxy error page is HTML.
4. **Building query strings by hand.** An `&`, `=` or space in a value silently changes the query. Pass `params=`.
5. **Using `data=` when you meant `json=`.** No `Content-Type` is set, and many APIs answer `400` or `415`.
6. 🔴 **Retrying a `POST` blindly.** It is not idempotent; you may create the thing twice (**18.3**).
7. **Treating `401` and `403` the same.** A refreshed token fixes one and never the other.
8. **Omitting `timeout=`.** Without it a request can hang forever — **11.5** covers why.
9. **Assuming the error body has your expected shape.** The 500 may come from a load balancer that has never heard of your API.

## Best Practices

- Call `raise_for_status()` on every response, then parse.
- Use `json=` to send and guard `.json()` when receiving.
- Always pass `params=`; never concatenate query strings.
- Set `Accept` and a descriptive `User-Agent` once, on a `Session`.
- Always pass `timeout=` — a connect and read pair, like `timeout=(3.05, 27)`.
- Let the status code drive the decision: 4xx means fix the request, 5xx means retry.
- Read the `Location` header after a `201` rather than guessing the new URL.
- Wrap the API in a small client class you own, so the rest of your code never sees `requests` (**15.5**, **16.5**).

## Practice Exercises

Try these before moving on.

1. Add a `HEAD` handler to the fake API and confirm `requests.head()` returns headers with no body. Why is `HEAD` useful?
2. 🔴 Write a `get_json(url)` helper that handles all four cases: 4xx, 5xx, `204`, and a non-JSON body. Test it against every endpoint in this notebook.
3. Send a value containing `&`, `=` and a space three ways: `params=`, an f-string, and `urllib.parse.urlencode`. Which survive?
4. Use the `/echo` endpoint to discover exactly what `requests` sends by default. Which headers did you not know were there?
5. Trigger each of `ConnectionError`, `ConnectTimeout`, `ReadTimeout` and `HTTPError` deliberately. Which of them means the server may still have done the work?
6. 🔴 Which of the five verbs would you retry automatically, and which never? Write the rule as code (**18.3** does exactly this).
7. **Interview question:** an API returns `200 OK` with `{"success": false}` in the body. What is wrong with that design, and how would you write a client for it anyway?

---

## Version notes

| Version | Change |
|---|---|
| **requests 2.32** | `JSONDecodeError` is now a `requests` exception, not `json`'s — catch `requests.exceptions.JSONDecodeError` |
| **Python 3.11** | `http.server` gained better HTTP/1.1 handling, used by the fake API here |
| **Python 3.9** | `dict |` merge, handy for building request payloads (**2.5**) |

> **`requests` vs `httpx` vs `urllib`.** `requests` is the most widely used and is what you will
> read. `httpx` has a nearly identical API plus **async** and HTTP/2 — **18.5** uses it for the
> concurrency section. `urllib.request` is the standard library and needs no install (**8.4**,
> **11.5**).

## Where next

| Notebook | Covers |
|---|---|
| **18.2** | authentication, tokens and handling secrets properly |
| **18.3** | pagination, rate limits, retries and backoff |
| **18.4** | validating what you receive, with `pydantic` |
| **18.5** | testing API clients, and concurrency for I/O-bound work |

## Related

- **11.5 HTTP** — the transport: sockets, `urllib`, timeouts, `Session`
- **8.3 JSON** — encoding and decoding, and what JSON cannot represent
- **8.4 Online Text** — reading over the network with the standard library
- **2.5 Dictionaries** — `.get()` and the `None` that follows (**16.2**)
- **15.7** — why raising beats returning a status you can ignore